# Transmon coupled to CPW resonator

> 💡 **Using this tutorial without the Qt GUI**
> 
> This tutorial uses the desktop `MetalGUI`. To follow along on Colab, Binder, JupyterHub, or any environment where Qt isn't available, **replace any `gui.rebuild()` / `gui.screenshot()` call with `qm.view(design)`** — it renders the design to a matplotlib `Figure` you can display inline or save with `fig.savefig(...)`.
> 
> See [1.4 Headless quick view](../../1-Overview/1.4-Headless-quick-view-%28no-Qt-GUI%29.ipynb) for a complete runnable walkthrough and [`docs/headless-usage.rst`](../../../docs/headless-usage.rst) for the full reference.

In [ ]:
%load_ext autoreload
%autoreload 2

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings
import pyEPR as epr

### Create the design in Metal
Setup a design of a given dimension. Dimensions will be respected in the design rendering. <br>
Note that the design size extends from the origin into the first quadrant.

In [ ]:
design = designs.DesignPlanar({}, True)
design.chips.main.size["size_x"] = "4mm"
design.chips.main.size["size_y"] = "6mm"

gui = MetalGUI(design)

Create one meander resonator connected to a transmons and open-to-ground qcomponents.

In [ ]:
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

In [ ]:
# Explore the options of the OpenToGround.
OpenToGround.get_template_options(design)

In [ ]:
# Explore the options of the TransmonPocket.
TransmonPocket.get_template_options(design)

In [ ]:
# Explore the options of the RouteMeander.
RouteMeander.get_template_options(design)

In [ ]:
# Setup the TransmonPocket loacation and add a connection_pad.
q1 = TransmonPocket(
    design,
    "Q1",
    options=dict(
        pad_width="425 um",
        pocket_height="650um",
        connection_pads=dict(readout=dict(loc_W=+1, loc_H=+1, pad_width="200um")),
    ),
)

In [ ]:
# Setup the OpenToGround location and orientation.
otg_options = dict(pos_x="2.5mm", pos_y="0.5mm", orientation="0")

otg = OpenToGround(design, "open_to_ground", options=otg_options)

In [ ]:
coupler_options = Dict(
    pin_inputs=Dict(
        start_pin=Dict(component=q1.name, pin="readout"),
        end_pin=Dict(component=otg.name, pin="open"),
    ),
    fillet="99.9um",
    total_length="5mm",
    lead=Dict(start_straight="200um"),
)


bus = RouteMeander(design, "coupler", options=coupler_options)

gui.rebuild()
gui.autoscale()

In [ ]:
# Get a list of all the qcomponents in QDesign and then zoom on them.
all_component_names = design.components.keys()

gui.zoom_on_components(all_component_names)

In [ ]:
# Save screenshot as a .png formatted file.
gui.screenshot()

In [ ]:
# Screenshot the canvas only as a .png formatted file.
gui.figure.savefig("shot.png")

from IPython.display import Image, display

_disp_ops = dict(width=500)
display(Image("shot.png", **_disp_ops))

In [ ]:
# Closing the Qiskit Metal GUI
gui.main_window.close()